# N-Gram Language Modeling with Unknown Words

| Assignment field | Details |
|---|---|
| **Group Number** | 90 |
| **Problem Statement** | 9 - N-gram Language Modeling with Unknown Words |
| **Dataset** | NLTK Gutenberg Corpus |
| **Student Name(s)** |  |
| **Course/Subject** |  |

## 1. Introduction

A **language model** assigns probabilities to sequences of words and can use them to predict likely words. An **N-gram** is a sequence of *N* consecutive tokens. A **trigram** model uses the previous two words to estimate the next word:

$$P(w_i \mid w_{i-2}, w_{i-1})$$

Words absent from the training vocabulary are a problem because the model has no counts for them. We use the special token `<UNK>` (unknown) to group rare training words and represent unseen test words. This gives the model a learned way to handle unfamiliar vocabulary.

## 2. Objective

This assignment will:

- build a trigram language model;
- analyze rare words and replace training words occurring fewer than 3 times with `<UNK>`;
- perform next-word and missing-word prediction;
- compare models with and without `<UNK>`;
- compare no smoothing with Laplace (add-one) smoothing; and
- evaluate accuracy, OOV rate, coverage, and perplexity.

## 3. Import libraries and reproducible setup

The setup checks for NLTK resources and downloads them only when absent. The fixed seed makes any sampling deterministic.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import math
import random
import re
import sys

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 90
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 90)

# Keep downloaded NLTK resources in an ignored local cache when possible.
NLTK_DATA_DIR = Path.cwd() / ".nltk_data"
NLTK_DATA_DIR.mkdir(exist_ok=True)
if str(NLTK_DATA_DIR) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_DATA_DIR))

required_resources = {
    "gutenberg": "corpora/gutenberg",
    "punkt": "tokenizers/punkt",
    "punkt_tab": "tokenizers/punkt_tab",
}
for package, resource_path in required_resources.items():
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(package, download_dir=str(NLTK_DATA_DIR), quiet=True, raise_on_error=True)

from nltk.corpus import gutenberg

print(f"Python: {sys.version.split()[0]}")
print(f"NLTK: {nltk.__version__}")
print(f"Random seed: {SEED}")

## 4. Load and inspect the NLTK Gutenberg Corpus

We first list every available file, then select one sufficiently large book. `austen-emma.txt` is suitable because it contains many sentences and repeated everyday vocabulary, while still containing rare words for the `<UNK>` experiment.

In [ ]:
available_books = gutenberg.fileids()
selected_book = "austen-emma.txt"

books_df = pd.DataFrame({"Available Gutenberg file": available_books})
display(books_df)
print(f"Number of books available: {len(available_books)}")
print(f"Selected book: {selected_book}")

In [ ]:
raw_text = gutenberg.raw(selected_book)
raw_sentences = list(gutenberg.sents(selected_book))
raw_tokens = list(gutenberg.words(selected_book))

corpus_stats_df = pd.DataFrame({
    "Statistic": ["Selected book", "Raw characters", "Raw sentences", "Tokens before preprocessing"],
    "Value": [selected_book, len(raw_text), len(raw_sentences), len(raw_tokens)],
})
display(corpus_stats_df)

## 5. Text preprocessing

Preprocessing decisions are explicit:

1. NLTK's Gutenberg sentence boundaries are retained.
2. Tokens are lowercased so `The` and `the` share counts.
3. Alphabetic words and internal apostrophes are kept; standalone punctuation and numbers are removed.
4. Two `<START>` tokens and one `<END>` token are added during trigram counting. Two starts are required because a trigram needs two preceding tokens.
5. `<UNK>` is **not** added yet; it is learned from training frequencies later.

In [ ]:
WORD_PATTERN = re.compile(r"^[A-Za-z]+(?:'[A-Za-z]+)?$")

def preprocess_sentence(tokens):
    # Lowercase and retain word-like tokens while preserving the sentence as a list.
    return [token.lower() for token in tokens if WORD_PATTERN.fullmatch(token)]

processed_sentences = [preprocess_sentence(sentence) for sentence in raw_sentences]
processed_sentences = [sentence for sentence in processed_sentences if sentence]

example_rows = []
for original, processed in zip(raw_sentences[:5], processed_sentences[:5]):
    example_rows.append({
        "Original sentence": " ".join(original[:30]),
        "After preprocessing": " ".join(processed[:30]),
    })
display(pd.DataFrame(example_rows))

## 6. Train/test data preparation

The corpus is split at sentence boundaries: the first 80% of sentences are training data and the remaining 20% are test data. This chronological, deterministic split prevents test sentences from influencing training counts or the `<UNK>` vocabulary.

In [ ]:
split_index = int(0.80 * len(processed_sentences))
train_sentences = processed_sentences[:split_index]
test_sentences = processed_sentences[split_index:]

split_df = pd.DataFrame({
    "Split": ["Training", "Testing", "Total"],
    "Sentences": [len(train_sentences), len(test_sentences), len(processed_sentences)],
    "Percentage": [100 * len(train_sentences) / len(processed_sentences),
                   100 * len(test_sentences) / len(processed_sentences), 100.0],
})
display(split_df.style.format({"Percentage": "{:.1f}%"}))